# Concept Portfolio V2 Lab

LIVE strict-schema 실패를 안전하게 진단하고 Plan Draft → system normalization → Candidate governance → Legal → downstream 계약을 단계별로 검증합니다. 기본 모드는 MOCK이며 이전 실행 output은 포함하지 않습니다.

## 00. Quick Start

처음에는 `MODE='MOCK'`로 Kernel Restart 후 Run All을 실행합니다. LIVE에서는 04 Schema Preflight부터 단계별로 진행하고, Plan·Candidate 1·Legal 1 결과를 각각 확인한 뒤 다음 호출을 실행합니다.

## 01. Environment

In [ ]:
import json, os, sys
from pathlib import Path

AI_ROOT = Path.cwd()
if AI_ROOT.name == 'notebooks':
    AI_ROOT = AI_ROOT.parent
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

from app.concept_portfolio_v2 import ConceptPortfolioEngine, ProviderGateway, ProviderMode
from app.concept_portfolio_v2.diagnostics.notebook_view import (
    show_seed_input, show_seed_analysis, show_design_space, show_portfolio_plans,
    show_plan_diversity, show_candidates, show_candidate_validation, show_legal_precheck,
    show_legal_result, show_redesign_diff, show_replan, show_final_portfolio,
    show_hypotheses, show_downstream_handoff, show_trace, show_provider_usage,
    show_schema_preflight, show_provider_failure, show_raw_json,
)
print('권장/production-equivalent: Python 3.12')
print('현재 kernel:', sys.version.split()[0], '(3.14.x는 local smoke이며 production-equivalent가 아닙니다)')

## 02. MODE

In [ ]:
MODE = 'MOCK'  # MOCK | REPLAY | LIVE
RECORDINGS_DIR = AI_ROOT / 'recordings' / 'concept_portfolio_v2'
gateway = ProviderGateway(ProviderMode(MODE), recordings_dir=RECORDINGS_DIR)
engine = ConceptPortfolioEngine(ProviderMode(MODE), gateway=gateway)
print('MODE =', MODE)
print('⚠ LIVE는 외부 Provider/MOLEG 호출 및 비용이 발생합니다.' if MODE == 'LIVE' else '외부 Provider 호출 없음')

## 03. LIVE env status

값 자체는 출력하지 않고 설정 여부만 표시합니다.

In [ ]:
LIVE_ENV_KEYS = ['AI_PROVIDER', 'AI_API_KEY', 'AI_MODEL', 'AI_BASE_URL', 'MOLEG_API_KEY', 'LEGAL_REGISTRY_VERSION']
{key: bool(os.getenv(key, '').strip()) for key in LIVE_ENV_KEYS}

## 04. Schema Preflight

Plan Draft Pool, 기존 accepted Candidate Draft, semantic distinctness/fidelity schema를 로컬에서 검사합니다. 기대값: **ALL PASS / Provider Calls 0**. 실패하면 Planning/Candidate 셀을 계속 실행하지 마십시오.

In [ ]:
schema_preflight = engine.schema_preflight_report()
show_schema_preflight(schema_preflight)
print('ALL PASS' if schema_preflight.status == 'PASS' else '❌ SCHEMA_PREFLIGHT_FAILED — 다음 셀을 실행하지 마십시오.')
print('Provider Calls:', schema_preflight.providerCalls)

## 05. Input

In [ ]:
TEST_INPUT = {
    'fixtureName': 'notebook_food_lab',
    'ideaOverview': '개인 맞춤형 식재료를 소량 제공하고 남은 재료 활용 레시피를 안내하는 서비스',
    'problem': '1~2인 가구가 식재료를 다 쓰지 못해 비용과 음식물 쓰레기가 발생한다',
    'targetUsers': '식재료 낭비를 줄이고 싶은 1~2인 가구',
    'price': {'value': '월 19,900원', 'decisionState': 'LOCKED', 'source': 'USER_INPUT'},
    'channels': {'value': '모바일 앱', 'decisionState': 'LOCKED', 'source': 'USER_INPUT'},
}

## 06. Seed Adapter

In [ ]:
engine._reset()
seed = engine.seed_adapter.adapt(TEST_INPUT)
show_seed_input(seed)

## 07. Safety

In [ ]:
safety = await engine.check_safety(seed)
safety.model_dump(mode='json')

## 08. Seed Analysis

In [ ]:
analysis = await engine.analyze_seed(seed) if safety.passed else None
show_seed_analysis(analysis) if analysis else 'Safety blocked'

## 09. Opportunity Anchors

Source LOCK과 business opportunity anchor는 별도 계약입니다. 하위 세그먼트 specialization은 허용하지만 무관한 고객 영역 drift는 거부합니다.

In [ ]:
analysis.opportunityAnchor.model_dump(mode='json') if analysis else {}

## 10. Open Design Space

In [ ]:
show_design_space(analysis) if analysis else []

## 11. Generate Plan Pool

LIVE Provider는 Draft-only strict schema를 반환합니다. `planId`, preserved anchors/locks는 요청하지 않습니다.

In [ ]:
PLANNING_READY = False
raw_plan_drafts = []
if schema_preflight.status == 'PASS' and analysis:
    try:
        raw_plan_drafts = await engine.generate_plan_drafts(seed, analysis, max_concepts=5)
        PLANNING_READY = True
        print('Plan drafts:', len(raw_plan_drafts))
    except Exception as failure:
        print('❌ LIVE PLANNING FAILED — No plan was generated. Do not continue to Candidate cells.')
        print(type(failure).__name__, str(failure))
        print(show_provider_failure(engine.gateway))
else:
    print('❌ PLANNING NOT EXECUTED — SCHEMA_PREFLIGHT_FAILED 또는 선행 단계 미완료')

## 12. Raw Provider Drafts / Normalized Plans

In [ ]:
normalized_plans = engine.normalize_plan_drafts(raw_plan_drafts, analysis) if PLANNING_READY else []
print('Provider Draft sample:', show_raw_json(raw_plan_drafts[0])[:2500] if raw_plan_drafts else '{}')
show_portfolio_plans(normalized_plans)

## 13. Plan Diversity

Level 1 canonical duplicate, Level 2 명확한 구조 차이, Level 3 ambiguous semantic judge 순서입니다.

In [ ]:
plan_validation = await engine.validate_plans(normalized_plans, analysis, max_concepts=5) if normalized_plans else None
show_plan_diversity(plan_validation.diversity) if plan_validation else []

## 14. Plan Selection

In [ ]:
selected_plans = plan_validation.acceptedPlans if plan_validation else []
show_portfolio_plans(selected_plans)

## 15. Expand Candidate 1 Only

LIVE 비용 통제 smoke입니다. 결과를 확인하기 전 remaining Candidates를 실행하지 마십시오.

In [ ]:
candidate_one = None
if selected_plans:
    try:
        candidate_one = await engine.expand_plan(seed, selected_plans[0], 1)
        print(show_raw_json(candidate_one)[:5000])
    except Exception as failure:
        print('❌ CANDIDATE 1 FAILED — remaining Candidate 셀을 실행하지 마십시오.')
        print(type(failure).__name__, str(failure), show_provider_failure(engine.gateway))

## 16. Candidate 1 Governance / Validation

31개 value semantics, direct user LOCK 보존, provenance, anchor, plan fidelity를 확인합니다.

In [ ]:
candidate_one_valid = []
candidate_one_reports = []
if candidate_one:
    candidate_one_valid, candidate_one_reports = await engine.validate_candidates(seed, selected_plans[:1], [candidate_one])
    print('value semantics:', len(candidate_one.candidate.valueSemantics))
show_candidate_validation(candidate_one_reports)

## 17. Expand Remaining Candidates

In [ ]:
remaining_candidates = []
if candidate_one_valid:
    for index, plan in enumerate(selected_plans[1:], 2):
        remaining_candidates.append(await engine.expand_plan(seed, plan, index))
expanded_all = [candidate_one] + remaining_candidates if candidate_one else []
candidates, candidate_reports = await engine.validate_candidates(seed, selected_plans, expanded_all) if expanded_all else ([], [])
show_candidates(candidates)

## 18. Candidate Pairwise Distinctness

In [ ]:
candidate_pairwise = [engine.compare_candidates(candidates[i], candidates[j])
                      for i in range(len(candidates)) for j in range(i + 1, len(candidates))]
show_plan_diversity(candidate_pairwise)

## 19. Legal Precheck Candidate 1

Structural risk precheck — not final legal review.

In [ ]:
precheck_one = engine.legal_precheck(candidates[0]) if candidates else None
show_legal_precheck([precheck_one]) if precheck_one else []

## 20. Full Legal Candidate 1

LIVE에서는 evidence, route, controls, redesign requirements를 확인한 후 다음 셀을 실행합니다.

In [ ]:
legal_one = await engine.review_legal_candidate(seed, candidates[0]) if candidates else None
show_legal_result([legal_one]) if legal_one else []

## 21. Full Legal Remaining

In [ ]:
legal_remaining = await engine.review_legal(seed, candidates[1:]) if legal_one and len(candidates) > 1 else []
legal_reviews = ([legal_one] if legal_one else []) + legal_remaining
show_legal_result(legal_reviews)

## 22. Redesign

동일 lineage별로 1회의 budget을 적용하고 child를 전체 Candidate validator에 다시 통과시킵니다.

In [ ]:
portfolio, legal_all, required_inputs, redesigned_count, replanned_count = (
    await engine.resolve_legal(seed, selected_plans, candidates, legal_reviews)
    if legal_reviews else ([], [], [], 0, 0)
)
redesign_children = [item for item in portfolio if item.parentCandidateId]
print('redesigned:', redesigned_count, 'required inputs:', required_inputs)
show_redesign_diff(next(item for item in candidates if item.candidateId == redesign_children[0].parentCandidateId), redesign_children[0]) if redesign_children else []

## 23. Replan

Replacement Plan은 Plan validation → Candidate expansion/governance → full validation/distinctness → Legal 순으로 재진입합니다.

In [ ]:
print('replanned:', replanned_count)
show_replan(type('PortfolioView', (), {'concepts': portfolio})()) if portfolio else []

## 24. Final Portfolio

In [ ]:
show_final_portfolio(type('PortfolioView', (), {'concepts': portfolio})()) if portfolio else []

## 25. Manual Concept Selection

In [ ]:
SELECTED_CANDIDATE_ID = portfolio[0].candidateId if portfolio else None  # 사용자가 직접 변경
selected_concept = next((item for item in portfolio if item.candidateId == SELECTED_CANDIDATE_ID), None)
SELECTED_CANDIDATE_ID

## 26. Hypothesis Proposal

In [ ]:
hypotheses = engine.build_or_load_current_hypothesis_contract(selected_concept) if selected_concept else []
show_hypotheses(hypotheses)

## 27. Hypothesis Confirmation

`auto_confirm_hypotheses`는 MOCK 회귀용 **Lab shortcut**이며 사용자 확인이나 production 결정을 뜻하지 않습니다. LIVE 기본값은 False입니다.

In [ ]:
AUTO_CONFIRM_HYPOTHESES = MODE == 'MOCK'  # Lab shortcut only; LIVE에서는 False
HYPOTHESIS_EDITS = {}  # 예: {'PRICE': '월 17,900원'}
if AUTO_CONFIRM_HYPOTHESES or HYPOTHESIS_EDITS:
    confirmed_hypotheses = engine.confirm_hypotheses(hypotheses, HYPOTHESIS_EDITS)
else:
    confirmed_hypotheses = hypotheses
    print('LIVE: 7개 hypothesis를 사용자가 확인하기 전 자동 확정하지 않습니다.')
show_hypotheses(confirmed_hypotheses)

## 28. Optional Delta Legal

법률 민감 hypothesis를 편집하면 delta legal이 필수입니다. 아래 집합은 실제 검토 완료 결과가 있을 때만 직접 입력합니다.

In [ ]:
DELTA_LEGAL_APPROVED_TYPES = set()  # 예: {'CHANNELS'}; 실제 delta legal 완료 후에만 입력
confirmed_hypotheses = engine.mark_delta_legal_reviewed(confirmed_hypotheses, DELTA_LEGAL_APPROVED_TYPES)
[item.hypothesisType for item in confirmed_hypotheses if item.deltaLegalRequired and item.legalReviewStatus != 'PASSED']

## 29. Market Seed

In [ ]:
handoff = None
if selected_concept:
    try:
        handoff = engine.build_downstream_handoff(seed, selected_concept, confirmed_hypotheses, legal_all)
    except Exception as failure:
        print('Handoff 준비 실패:', str(failure))
handoff.marketAnalysisSeedSnapshot if handoff else {}

## 30. Marketing Handoff

In [ ]:
handoff.marketingSourceSnapshot if handoff else {}

## 31. Contract Compatibility

In [ ]:
show_downstream_handoff(handoff) if handoff else {'계약': 'NOT_READY'}

## 32. Trace

In [ ]:
show_trace(engine.trace)

## 33. Provider Usage

논리 작업과 실제 외부 Provider 호출을 분리합니다.

In [ ]:
show_provider_usage(engine.gateway.usage)

## 34. Record / Replay

LIVE 기록은 canonical request hash에 결합되고 비밀키/Authorization/token은 재귀적으로 redaction됩니다. REPLAY miss는 MOCK으로 대체하지 않습니다.

In [ ]:
print('recordings:', RECORDINGS_DIR)
print('mode:', MODE, 'last safe failure:', show_provider_failure(engine.gateway))

## 35. One-click MOCK Run

In [ ]:
mock_result = await ConceptPortfolioEngine('MOCK').run_full(
    TEST_INPUT, max_concepts=5, auto_confirm_hypotheses=True)  # Lab shortcut
print(mock_result.runStatus.value, mock_result.downstreamReadiness)

## 36. One-click LIVE Run

위의 staged LIVE smoke가 모두 성공한 뒤 마지막에만 명시적으로 활성화합니다.

In [ ]:
RUN_ONE_CLICK_LIVE = False
if MODE == 'LIVE' and RUN_ONE_CLICK_LIVE:
    live_result = await ConceptPortfolioEngine('LIVE').run_full(
        TEST_INPUT, max_concepts=5, auto_confirm_hypotheses=False)
    print(live_result.runStatus.value, live_result.downstreamReadiness)
else:
    print('One-click LIVE 비활성화 — staged smoke 완료 후 RUN_ONE_CLICK_LIVE=True로 변경')

## 37. Known Limitations

- MOCK 성공은 LIVE Provider schema acceptance나 법률 결론을 증명하지 않습니다.
- Legal structural precheck는 최종 법률검토가 아닙니다.
- LIVE Legal도 법률 자문이 아닙니다.
- Canonical Notebook은 output과 execution count를 비운 상태로 유지합니다.
- Provider response-format/schema permanent failure는 재시도하지 않습니다. 429, timeout, network, 5xx만 제한적으로 재시도합니다.